In [10]:
import torch
from transformers import pipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [5]:
SEED = 42
torch.manual_seed(6033689)
model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading model: {model_id}")

Loading model: Qwen/Qwen2.5-7B-Instruct


In [13]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

system_prompt = (
    "You are an expert Speech-Language Pathologist creating a script training dataset for aphasia rehabilitation. "
    "Your task is to generate exactly 25 distinct sentences that a patient might practice during functional therapy. "
    "These should cover daily living scenarios like ordering food, speaking to a doctor, talking to family, or navigating transportation."
)

user_constraints = (
    "Generate the 25 functional therapy sentences based on the following strict constraints:\n"
    "1. Length: Every single sentence MUST be between 9 and 12 words long. Count the words carefully.\n"
    "2. Tone: Must be natural, conversational, spoken English. Avoid complex, multi-clause academic sentences.\n"
    "3. Sentence Type: Every sentence MUST be a declarative statement. Do NOT generate any questions or use question marks.\n"
    "4. Perspective: These are statements spoken BY the patient to declare their needs, feelings, observations, or actions.\n"
    "5. Format: Output only a numbered list of the 25 sentences. Do not include any introductory or concluding text."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_constraints},
]

print("Generating aphasia therapy scripts...\n")
print("-" * 50)

outputs = generator(
    messages,
    max_new_tokens=1000,
    temperature=0.3,
    do_sample=True,
    pad_token_id=generator.tokenizer.eos_token_id
)

generated_text = outputs[0]["generated_text"][-1]["content"]
print(generated_text)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating aphasia therapy scripts...

--------------------------------------------------
1. I need a slice of pizza for dinner tonight.
2. Can you help me find a bus stop nearby?
3. I want to order a salad with chicken for lunch.
4. My appointment is at 3 PM this afternoon.
5. Could you please pass the salt, please?
6. I'm feeling better today, thank you.
7. I need to pick up some groceries after work.
8. The train leaves from platform number five.
9. I'd like to book a table for two, please.
10. My mom will be here in an hour.
11. I need to take my medication every day.
12. The doctor said I should get more sleep.
13. I want to go for a walk in the park.
14. My sister is coming over to visit tomorrow.
15. I need to buy some new clothes soon.
16. The movie starts at 7 PM sharp.
17. I need to fill out this form correctly.
18. My friend is picking me up at 6 PM.
19. I want to order a large pizza for delivery.
20. The pharmacy is just around the corner.
21. I need to pay my bills next we